# Silver template: clean, key and upsert one table (Databricks)

Reads one Bronze table, cleans it to an explicit contract, gives every row its surrogate key from the
central key table, and upserts it into Silver. Copy this notebook, fill in the **Parameters** cell, and run it.

**Standards this notebook follows**

1. **Layers.** one Unity Catalog catalog per layer: `bronze`, `silver`, `gold`, with keys in `keys`. Tables are named `<catalog>.<schema>.<table>`.
2. **Contract.** Only the columns in `columns` are kept, each cast to its type. No `SELECT *`, so a new
   source column never leaks into Silver unreviewed.
3. **Surrogate keys.** `<key_table>_sk` comes from the key table `keys.sk.<key_table>`: the same
   business key gets the same key from every source, and a rebuild never renumbers it. The key table's `<name>_sk` is a `BIGINT GENERATED BY DEFAULT AS IDENTITY` column, so Databricks issues new keys and the `MERGE` only inserts business keys. (Identity columns must be `BIGINT`; `BY DEFAULT` lets the keys admin seed the Unknown member, `-1`.) A table with an identity column takes one writer at a time, so the `MERGE` is retried on a concurrent write.
4. **No key, no row.** Rows with a missing business key go to `<target>_quarantine` with a reason.
5. **Clean.** Text is trimmed, blank text becomes NULL, chosen columns are lower-cased, and the newest
   row per business key wins (by the watermark).
6. **Incremental.** Reads rows with `watermark >= ` the newest one already in Silver (`>=`, not `>`, so rows
   that share the last timestamp aren't lost); the MERGE makes re-reading them harmless.
7. **Upsert, never overwrite.** `MERGE` on the surrogate key; a row is updated only when its `_row_hash`
   changed and its watermark isn't older than the stored one.
8. **Audit columns** on every row: `_row_hash`, `_run_id`, `_loaded_at`.
9. Passes the app's notebook best-practice rules (no secrets, ids in paths, `collect()` or writes in loops).

**Before the first run** a keys admin creates the key table once (this notebook only adds business keys to it):

```sql
CREATE TABLE IF NOT EXISTS keys.sk.customer (
    customer_sk BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 1 INCREMENT BY 1),
    customer_bk STRING    NOT NULL,
    create_date TIMESTAMP NOT NULL
);
INSERT INTO keys.sk.customer (customer_sk, customer_bk, create_date) VALUES (-1, 'UNKNOWN', current_timestamp());
```

## Who supports this notebook

This is an **enterprise template** (`silver-upsert` 1.0.0, Databricks). The line between enterprise and domain is drawn by the cells:

| Cells | Marked | Who owns them |
| --- | --- | --- |
| **Parameters** | `# kratosforge:parameters` | The **domain**: table names, keys, columns, measures. Values only, no code. |
| Every other code cell | `# kratosforge:standard` | The **enterprise**: it maintains, fixes and upgrades them. Don't edit them. |
| Markdown cells | none | Anyone: add notes freely. |

While the standard cells are unchanged and no code cell is added, the notebook **conforms** and the enterprise can
take it over: check it with `fabric-desktop templates check <notebook>`, then hand it over with
`fabric-desktop templates handover <notebook> --domain <domain> --owner <you>`.
Editing a standard cell, or adding a code cell, makes it **domain-owned** ad hoc work: it still runs, but the
domain supports it, and the check lists each change. Need logic the template doesn't have? Put it in Silver
(a derived column) or ask the enterprise to add it to the template, so every domain gets it.

In [ ]:
# kratosforge:parameters template=silver-upsert platform=databricks version=1.0.0 | The domain fills in this cell: values only, no code.
# Parameters
source_table = "bronze.crm.customers"
target_table = "silver.crm.customers"
key_table = "keys.sk.customer"           # issues customer_sk for customer_bk
business_key = ["customer_id"]               # the columns that identify a row in the source system
watermark_column = "updated_at"              # when the source row last changed
columns = {                                  # the Silver contract: every column kept, and its type
    "customer_id": "int",
    "name": "string",
    "segment": "string",
    "city": "string",
    "email": "string",
    "updated_at": "date",
}
lowercase_columns = ["email"]
run_id = ""          # the pipeline or orchestrator passes its run id; one is made when empty
full_reload = False  # True re-reads the whole source (after a logic change or a backfill)

In [ ]:
# kratosforge:standard template=silver-upsert platform=databricks version=1.0.0 cell=setup | Enterprise-managed: change the Parameters cell, not this one.
# Job runs pass parameters to a notebook task as widgets; a widget with a parameter's name overrides it.
for _name in ("run_id", "full_reload", "load_date"):
    if _name in globals():
        dbutils.widgets.text(_name, str(globals()[_name]))
        _value = dbutils.widgets.get(_name)
        globals()[_name] = _value.strip().lower() == "true" if isinstance(globals()[_name], bool) else _value

import re
import time
import uuid
from datetime import datetime, timezone
from functools import reduce

from pyspark.errors import AnalysisException
from pyspark.sql import Window
from pyspark.sql import functions as F

UNKNOWN_SK = -1  # the Unknown member: rows whose key can't be found point here, never at NULL
_TABLE_PART = re.compile(r"[A-Za-z_][A-Za-z0-9_ -]{0,127}")
_COLUMN = re.compile(r"[A-Za-z_][A-Za-z0-9_]{0,127}")
_TYPE = re.compile(r"(?i)(string|boolean|tinyint|smallint|int|bigint|float|double|date|timestamp|decimal\(\d{1,2},\s*\d{1,2}\))")
_RUN_ID = re.compile(r"[A-Za-z0-9_.:-]{1,64}")


def q(name: str) -> str:
    """A table name with each part in backticks. Parameters arrive from pipelines, so they are checked."""
    parts = name.split(".")
    if not 1 <= len(parts) <= 4 or not all(_TABLE_PART.fullmatch(x) for x in parts):
        raise ValueError(f"Not a table name: {name!r}")
    return ".".join(f"`{x}`" for x in parts)


def c(name: str) -> str:
    if not _COLUMN.fullmatch(name):
        raise ValueError(f"Not a column name: {name!r}")
    return f"`{name}`"


def checked_type(dtype: str) -> str:
    if not _TYPE.fullmatch(dtype.strip()):
        raise ValueError(f"Not a supported type: {dtype!r}")
    return dtype.strip()


def table_exists(name: str) -> bool:
    try:
        spark.table(q(name))  # resolved now, so a missing table raises here
        return True
    except AnalysisException:
        return False


def row_hash(columns: list) -> "F.Column":
    """SHA-256 of the columns, NULL-safe: a change in any of them changes the hash."""
    parts = [F.coalesce(F.col(x).cast("string"), F.lit("\u0000")) for x in columns]
    return F.sha2(F.concat_ws("\u001f", *parts), 256)


run_id = run_id or str(uuid.uuid4())
if not _RUN_ID.fullmatch(run_id):
    raise ValueError("run_id may only hold letters, digits and _ . : -")
loaded_at = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"Run {run_id} at {loaded_at:%Y-%m-%d %H:%M:%S} UTC")

## 1. Read what changed

In [ ]:
# kratosforge:standard template=silver-upsert platform=databricks version=1.0.0 cell=step-1 | Enterprise-managed: change the Parameters cell, not this one.
key_name = key_table.rsplit(".", 1)[-1]
sk_column, bk_column = f"{key_name}_sk", f"{key_name}_bk"
quarantine_table = f"{target_table}_quarantine"
for name in [*business_key, watermark_column, *lowercase_columns]:
    if name not in columns:
        raise ValueError(f"{name} must be one of the columns in the contract")
columns = {c_: checked_type(t) for c_, t in columns.items() if c(c_)}

high_water = None
if table_exists(target_table) and not full_reload:
    high_water = spark.table(q(target_table)).agg(F.max(c(watermark_column))).first()[0]

incoming = spark.table(q(source_table)).select(*[F.col(c(n)).cast(t).alias(n) for n, t in columns.items()])
if high_water is not None:
    incoming = incoming.where(F.col(c(watermark_column)) >= F.lit(high_water))
print(f"Reading {source_table}" + (f" from {watermark_column} {high_water}" if high_water is not None else " in full"))

## 2. Clean to the contract

In [ ]:
# kratosforge:standard template=silver-upsert platform=databricks version=1.0.0 cell=step-2 | Enterprise-managed: change the Parameters cell, not this one.
def cleaned(name: str, dtype: str) -> "F.Column":
    if dtype.lower() != "string":
        return F.col(c(name))
    text = F.trim(F.col(c(name)))
    text = F.lower(text) if name in lowercase_columns else text
    return F.when(text == "", F.lit(None)).otherwise(text).alias(name)


incoming = incoming.select(*[cleaned(n, t) for n, t in columns.items()])

## 3. Quarantine rows with no business key, then keep the newest row per key

In [ ]:
# kratosforge:standard template=silver-upsert platform=databricks version=1.0.0 cell=step-3 | Enterprise-managed: change the Parameters cell, not this one.
missing_key = reduce(lambda a, b: a | b, [F.col(c(k)).isNull() for k in business_key])
rejected = incoming.where(missing_key)
if rejected.limit(1).count():
    (rejected.withColumn("_reason", F.lit("business key missing"))
        .withColumn("_run_id", F.lit(run_id))
        .withColumn("_loaded_at", F.lit(loaded_at).cast("timestamp"))
        .write.format("delta").mode("append").option("mergeSchema", "true")
        .saveAsTable(q(quarantine_table)))
    print(f"Some rows had no business key; they are in {quarantine_table}")

# The newest row per business key wins.
newest = Window.partitionBy(*[F.col(c(k)) for k in business_key]).orderBy(F.col(c(watermark_column)).desc_nulls_last())
staged = (incoming.where(~missing_key)
          .withColumn("_rn", F.row_number().over(newest)).where(F.col("_rn") == 1).drop("_rn")
          .persist())
staged.createOrReplaceTempView("silver_staged")

## 4. Surrogate keys from the key table

In [ ]:
# kratosforge:standard template=silver-upsert platform=databricks version=1.0.0 cell=step-4 | Enterprise-managed: change the Parameters cell, not this one.
if not table_exists(key_table):
    raise RuntimeError(f"{key_table} doesn't exist yet. A keys admin creates it once (see the first cell).")


def bk_sql(alias: str) -> str:
    """The business key as one string: trimmed, blank as NULL. Composite keys join with || and escape |."""
    parts = [f"NULLIF(TRIM(CAST({alias}.{c(k)} AS STRING)), '')" for k in business_key]
    if len(parts) == 1:
        return parts[0]
    parts = [f"REPLACE({x}, '|', CONCAT(CHR(92), '|'))" for x in parts]
    return "CONCAT(" + ", '||', ".join(parts) + ")"


# 1. New business keys are added; the identity column gives them their keys. Existing ones keep theirs.
merge_new_keys = f"""
MERGE INTO {q(key_table)} AS k
USING (SELECT DISTINCT {bk_sql('b')} AS bk FROM silver_staged b) AS s
ON k.{c(bk_column)} = s.bk
WHEN NOT MATCHED AND s.bk IS NOT NULL THEN
    INSERT ({c(bk_column)}, create_date) VALUES (s.bk, current_timestamp())
"""

for attempt in range(1, 4):
    try:
        spark.sql(merge_new_keys)
        break
    except Exception as error:  # another run added keys at the same moment: read the table again and retry
        if attempt == 3 or "concurrent" not in f"{type(error).__name__} {error}".lower():
            raise
        time.sleep(5 * attempt)

# 2. Every row takes its key. A row that still has none would be a bug, so the run stops.
keyed = spark.sql(f"""
SELECT CAST(COALESCE(k.{c(sk_column)}, {UNKNOWN_SK}) AS BIGINT) AS {c(sk_column)},
       {", ".join(f"b.{c(n)}" for n in columns)}
FROM silver_staged b
LEFT JOIN {q(key_table)} k ON k.{c(bk_column)} = {bk_sql('b')}
""")
if keyed.where(F.col(c(sk_column)) == UNKNOWN_SK).limit(1).count():
    raise RuntimeError(f"Some rows found no key in {key_table}; nothing was written")

## 5. Upsert into Silver

In [ ]:
# kratosforge:standard template=silver-upsert platform=databricks version=1.0.0 cell=step-5 | Enterprise-managed: change the Parameters cell, not this one.
audit = ["_row_hash", "_run_id", "_loaded_at"]
keyed = (keyed.withColumn("_row_hash", row_hash(list(columns)))
              .withColumn("_run_id", F.lit(run_id))
              .withColumn("_loaded_at", F.lit(loaded_at).cast("timestamp")))
keyed.createOrReplaceTempView("silver_keyed")

if not table_exists(target_table):
    ddl = ", ".join([f"{c(sk_column)} BIGINT NOT NULL",
                     *[f"{c(n)} {t}" for n, t in columns.items()],
                     "`_row_hash` STRING", "`_run_id` STRING", "`_loaded_at` TIMESTAMP"])
    spark.sql(f"CREATE TABLE IF NOT EXISTS {q(target_table)} ({ddl}) USING DELTA CLUSTER BY ({c(sk_column)})")

every = [sk_column, *columns, *audit]
wm = c(watermark_column)
result = spark.sql(f"""
MERGE INTO {q(target_table)} AS t
USING silver_keyed AS s
ON t.{c(sk_column)} = s.{c(sk_column)}
WHEN MATCHED AND t._row_hash <> s._row_hash AND (t.{wm} IS NULL OR s.{wm} >= t.{wm}) THEN
    UPDATE SET {", ".join(f"t.{c(n)} = s.{c(n)}" for n in every if n != sk_column)}
WHEN NOT MATCHED THEN
    INSERT ({", ".join(c(n) for n in every)}) VALUES ({", ".join(f"s.{c(n)}" for n in every)})
""").first()
print(f"Silver MERGE: {result.asDict() if result else 'done'}")

## 6. Check the result

In [ ]:
# kratosforge:standard template=silver-upsert platform=databricks version=1.0.0 cell=step-6 | Enterprise-managed: change the Parameters cell, not this one.
summary = spark.sql(f"""
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT {c(sk_column)}) AS key_count,
       SUM(CASE WHEN {c(sk_column)} = {UNKNOWN_SK} THEN 1 ELSE 0 END) AS unknown_count
FROM {q(target_table)}
""").first()
if summary["row_count"] != summary["key_count"] or summary["unknown_count"]:
    raise RuntimeError(f"{target_table} has duplicate or Unknown keys: {summary.asDict()}")
print(f"{target_table}: {summary['row_count']} rows, one per {sk_column}")
staged.unpersist()